# Parte 2 · Del LLM al clasificador

**Fuente:** `datos/noticias.csv` generado en Parte 1. **Etiquetas de referencia:** secciones editoriales normalizadas. El test se separa al inicio y permanece fuera del ajuste, la auditoría y la selección de hiperparámetros. Las métricas reportadas indican siempre el conjunto usado. `datos/etiquetas_llm.csv` debe provenir de una ejecución real de `etiquetar_llm.py`; no se acepta un archivo de ejemplo.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

RAIZ = Path.cwd()
CLASES = ['politica', 'economia', 'deportes', 'ciencia_tecnologia']
df = pd.read_csv(RAIZ / 'datos/noticias.csv', encoding='utf-8-sig')
assert len(df) >= 350 and df['etiqueta'].isin(CLASES).all()
assert df['titular'].is_unique, 'Revisar títulos duplicados antes del split.'


## Split estratificado 70 / 15 / 15

Primero se reserva 15 % para test. El 85 % restante se divide en 70 % de entrenamiento y 15 % de validación respecto al total. Se estratifica en ambos pasos y se fija `random_state=42`. Solo se muestran tamaños y distribución; ningún titular de test se usa aquí.


In [ ]:
train_val, test = train_test_split(df, test_size=0.15, stratify=df['etiqueta'], random_state=42)
train, val = train_test_split(train_val, test_size=0.15/0.85,
                              stratify=train_val['etiqueta'], random_state=42)
assert not (set(train['url']) & set(val['url']))
assert not (set(train['url']) & set(test['url']))
assert not (set(val['url']) & set(test['url']))
print('Train, Val, Test:', len(train), len(val), len(test))
display(pd.DataFrame({k: x['etiqueta'].value_counts().reindex(CLASES, fill_value=0)
                      for k, x in [('train', train), ('val', val), ('test', test)]}))


## Línea base zero-shot: 40 ejemplos de validación

Esta primera medición sirve como piso antes de entrenar y preserva el test. La comparación final repetirá el zero-shot sobre 40 casos de test, después del entrenamiento, para permitir una comparación en la misma muestra.


In [ ]:
from transformers import pipeline
candidatos = {
    'politica': 'política', 'economia': 'economía', 'deportes': 'deportes',
    'ciencia_tecnologia': 'ciencia y tecnología'
}
inverso = {v: k for k, v in candidatos.items()}
zero_shot = pipeline('zero-shot-classification', model='Recognai/bert-base-spanish-wwm-cased-xnli')
muestra_val = val.sample(n=40, random_state=42)
def predecir_zero_shot(titulares):
    return [inverso[zero_shot(t, candidate_labels=list(inverso),
                               hypothesis_template='Esta noticia trata de {}.')['labels'][0]]
            for t in titulares]
pred_zs_val = predecir_zero_shot(muestra_val['titular'])
print('Zero-shot validación (n=40): accuracy =',
      round(accuracy_score(muestra_val['etiqueta'], pred_zs_val), 3),
      '| F1 macro =', round(f1_score(muestra_val['etiqueta'], pred_zs_val,
                                  labels=CLASES, average='macro'), 3))


## Supervisión débil y auditoría humana

Ejecuta previamente `python etiquetar_llm.py --modelo qwen2.5:3b`. El programa pide una etiqueta y confianza en JSON para cada titular y guarda `titular`, `etiqueta_llm`, `confianza`, `modelo_llm`, `fecha`. La confianza es una autoevaluación del LLM. Se extraen 20 casos aleatorios de entrenamiento para auditoría; una persona debe llenar `etiqueta_humana` antes de ver las predicciones del LLM. Los blancos no cuentan como acuerdo.


In [ ]:
llm = pd.read_csv(RAIZ / 'datos/etiquetas_llm.csv', encoding='utf-8-sig')
assert set(['titular','etiqueta_llm','confianza','modelo_llm','fecha']).issubset(llm)
assert len(llm) == len(df) and llm['titular'].is_unique
assert llm['etiqueta_llm'].isin(CLASES).all()
assert llm['confianza'].between(0, 1).all()
train_llm = train.merge(llm, on='titular', validate='one_to_one')
val_llm = val.merge(llm, on='titular', validate='one_to_one')
assert len(train_llm) == len(train) and len(val_llm) == len(val)
muestra_auditoria = train_llm.sample(n=20, random_state=42)[['titular']].copy()
muestra_auditoria['etiqueta_humana'] = ''
plantilla = RAIZ / 'datos/auditoria_humana_plantilla.csv'
plantilla_respuesta = RAIZ / 'datos/auditoria_humana_respuesta.csv'
muestra_auditoria.to_csv(plantilla, index=False, encoding='utf-8-sig')
print('Plantilla creada:', plantilla)
print('Copia la plantilla a', plantilla_respuesta, 'y rellena etiqueta_humana para los 20 casos.')


In [ ]:
if plantilla_respuesta.exists():
    audit = pd.read_csv(plantilla_respuesta, encoding='utf-8-sig').fillna('')
    assert len(audit) == 20 and audit['titular'].is_unique
    assert set(audit['titular']) == set(muestra_auditoria['titular'])
    assert audit['etiqueta_humana'].isin(CLASES).all(), 'Faltan etiquetas humanas válidas.'
    audit = audit.merge(train_llm[['titular', 'etiqueta_llm']], on='titular', validate='one_to_one')
    aciertos = int((audit['etiqueta_humana'] == audit['etiqueta_llm']).sum())
    acuerdo = aciertos / 20
    print(f'Acuerdo LLM-humano: {aciertos}/20 = {acuerdo:.1%}')
    display(audit[['titular','etiqueta_llm','etiqueta_humana']])
else:
    aciertos = None
    acuerdo = None
    print('Auditoría pendiente: se requiere revisión humana independiente de 20 casos.')


## Fine-tuning de DistilBERT multilingüe

El entrenamiento utiliza **solo** titulares y etiquetas del LLM del conjunto train. Validación emplea la categoría editorial como referencia, optimiza F1 macro y elige el mejor checkpoint de tres épocas. El modelo se guarda en `modelo/`; pesos grandes se excluyen de Git. Con 350–400 titulares la varianza entre particiones puede ser considerable.


In [ ]:
import torch
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, Trainer, TrainingArguments)
torch.manual_seed(42)
MODELO_BASE = 'distilbert/distilbert-base-multilingual-cased'
tokenizador = AutoTokenizer.from_pretrained(MODELO_BASE)
id2label = {i: c for i, c in enumerate(CLASES)}
label2id = {c: i for i, c in id2label.items()}
modelo = AutoModelForSequenceClassification.from_pretrained(
    MODELO_BASE, num_labels=4, id2label=id2label, label2id=label2id)

class Titulares(torch.utils.data.Dataset):
    def __init__(self, textos, etiquetas):
        self.codigos = tokenizador(list(textos), truncation=True, max_length=128)
        self.etiquetas = [label2id[e] for e in etiquetas]
    def __len__(self):
        return len(self.etiquetas)
    def __getitem__(self, i):
        return {**{k: v[i] for k, v in self.codigos.items()}, 'labels': self.etiquetas[i]}

def metricas_val(evaluacion):
    logits, etiquetas = evaluacion
    pred = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(etiquetas, pred),
            'f1_macro': f1_score(etiquetas, pred, labels=range(4), average='macro')}

argumentos = TrainingArguments(
    output_dir=str(RAIZ / 'modelo' / 'checkpoints'),
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy='epoch', save_strategy='epoch',
    load_best_model_at_end=True, metric_for_best_model='f1_macro',
    greater_is_better=True, save_total_limit=2,
    seed=42, report_to='none',
)
trainer = Trainer(
    model=modelo, args=argumentos,
    train_dataset=Titulares(train_llm['titular'], train_llm['etiqueta_llm']),
    eval_dataset=Titulares(val['titular'], val['etiqueta']),
    processing_class=tokenizador,
    data_collator=DataCollatorWithPadding(tokenizador),
    compute_metrics=metricas_val,
)
trainer.train()
trainer.save_model(str(RAIZ / 'modelo'))
tokenizador.save_pretrained(RAIZ / 'modelo')


## Evaluación final sobre test reservado

Solo ahora se leen titulares y etiquetas de test para evaluación. La muestra común de 40 permite comparar los tres enfoques bajo las mismas observaciones. Se reportan además métricas del LLM y Transformer sobre **todo** el test, con matriz de confusión y análisis por categoría.


In [ ]:
predicciones = trainer.predict(Titulares(test['titular'], test['etiqueta']))
pred_test = [id2label[i] for i in np.argmax(predicciones.predictions, axis=-1)]
y_test = test['etiqueta'].tolist()
exactitud = float(accuracy_score(y_test, pred_test))
f1 = float(f1_score(y_test, pred_test, labels=CLASES, average='macro'))
print(f'Transformer TEST (n={len(test)}): accuracy={exactitud:.3f}, F1 macro={f1:.3f}')
print(classification_report(y_test, pred_test, labels=CLASES, zero_division=0))
matriz = confusion_matrix(y_test, pred_test, labels=CLASES)
plt.figure(figsize=(9, 7))
sns.heatmap(matriz, annot=True, fmt='d', cmap='Blues', xticklabels=CLASES, yticklabels=CLASES)
plt.xlabel('Predicción'); plt.ylabel('Categoría editorial'); plt.title('Errores del Transformer en el test reservado')
plt.tight_layout(); plt.show()


In [ ]:
test_llm = test.merge(llm, on='titular', validate='one_to_one')
assert len(test_llm) == len(test)
f1_llm = float(f1_score(test_llm['etiqueta'], test_llm['etiqueta_llm'], labels=CLASES, average='macro'))
print(f'LLM débil TEST completo (n={len(test)}): accuracy=',
      round(accuracy_score(test_llm['etiqueta'], test_llm['etiqueta_llm']), 3),
      '| F1 macro =', round(f1_llm, 3))
muestra_test = test.sample(n=40, random_state=42)
pred_zs_test = predecir_zero_shot(muestra_test['titular'])
f1_zs = float(f1_score(muestra_test['etiqueta'], pred_zs_test, labels=CLASES, average='macro'))
pred_modelo_40 = [pred_test[test.index.get_loc(i)] for i in muestra_test.index]
muestra_llm = muestra_test.merge(llm, on='titular', validate='one_to_one')
comparativa = pd.DataFrame([
    {'método': 'Zero-shot', 'conjunto': 'test común', 'n': 40,
     'accuracy': accuracy_score(muestra_test['etiqueta'], pred_zs_test), 'F1 macro': f1_zs},
    {'método': 'LLM débil', 'conjunto': 'test común', 'n': 40,
     'accuracy': accuracy_score(muestra_llm['etiqueta'], muestra_llm['etiqueta_llm']),
     'F1 macro': f1_score(muestra_llm['etiqueta'], muestra_llm['etiqueta_llm'], labels=CLASES, average='macro')},
    {'método': 'Transformer ajustado', 'conjunto': 'test común', 'n': 40,
     'accuracy': accuracy_score(muestra_test['etiqueta'], pred_modelo_40),
     'F1 macro': f1_score(muestra_test['etiqueta'], pred_modelo_40, labels=CLASES, average='macro')},
])
display(comparativa)


In [ ]:
informe = classification_report(y_test, pred_test, labels=CLASES, output_dict=True, zero_division=0)
por_clase = {c: informe[c]['f1-score'] for c in CLASES}
mejor = max(por_clase, key=por_clase.get)
peor = min(por_clase, key=por_clase.get)
print('Mejor categoría:', mejor, round(por_clase[mejor], 3))
print('Peor categoría:', peor, round(por_clase[peor], 3))
print('Revisar los casos confundidos para explicar si hay vocabulario compartido o temas transversales.')
errores = test.assign(prediccion=pred_test)
display(errores[errores['etiqueta'] != errores['prediccion']][['titular','etiqueta','prediccion']].head(15))
resultados = {
    'test': {'n': len(test), 'accuracy': exactitud, 'f1_macro': f1},
    'zero_shot_test_40': {'accuracy': float(accuracy_score(muestra_test['etiqueta'], pred_zs_test)),
                          'f1_macro': f1_zs},
    'llm_test': {'n': len(test), 'f1_macro': f1_llm},
    'auditoria': {'aciertos': aciertos, 'total': 20, 'acuerdo': acuerdo},
    'por_clase': por_clase,
}
(RAIZ / 'modelo/metrics.json').write_text(json.dumps(resultados, indent=2, ensure_ascii=False), encoding='utf-8')


## Decisión para un entorno real

Tras ejecutar, comparar la tabla y revisar la latencia. Zero-shot no requiere entrenar, pero infiere con un modelo grande en cada solicitud. El LLM exige un servicio de generación y su confianza no está calibrada. DistilBERT tiene costo inicial de ajuste, pero inferencia local y rápida; elegirlo **solo si** el F1 en test es suficiente y se revisan los errores por categoría. Escribir aquí la decisión concreta sustentada por los números y al menos dos errores observados.

    **Reflexión por categoría:** pendiente de resultados reales. **Decisión final:** pendiente de ejecución y auditoría humana.
